# 06 — Focused classification audit

Stream C, tasks **C6–C8**. This notebook reports the final classifier's held-out performance by training-frequency group, clause length, and rule-detected party-role proxies, then runs a bounded counterfactual stress test on a reproducible sample of test clauses.

**Run all cells from top to bottom** (or use Restart Kernel → Run All). Later cells depend on the data and prediction variables loaded in section 1.

It requires the final Legal-BERT test prediction file and checkpoint. The counterfactuals are heuristic text edits, not legal redrafts; inspect examples and describe results as exploratory. Party roles are keyword-based proxies, not contract metadata. This focused audit does not claim coverage of all parties, contract types, European-language contracts, or retrieval robustness.


## 1. Load the fixed test data and prediction files

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # If the notebook runs from notebooks/
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

from src.audit.classifier import HuggingFaceTextPredictor
from src.audit.perturb import (
    audit_counterfactuals,
    generate_counterfactual_pairs,
    summarize_counterfactuals,
)
from src.audit.slices import build_audit_frame, slice_performance
from src.config import PREDICTIONS_DIR, RESULTS_DIR, RUNS_DIR
from src.data import load_ledgar, to_pandas
from src.utils import set_seed

set_seed(42)
dataset = load_ledgar()
train_df = to_pandas(dataset, "train")
test_df = to_pandas(dataset, "test")

# Change the run IDs if your teammates used different names.
FINAL_MODEL_RUN_ID = "legal-bert_final_20261019-1523"
TFIDF_RUN_ID = ""
final_prediction_path = PREDICTIONS_DIR / f"{FINAL_MODEL_RUN_ID}_test.parquet"
if not final_prediction_path.exists():
    raise FileNotFoundError(
        f"Missing final-model predictions: {final_prediction_path}. "
        "Copy the common-format test prediction file into results/predictions/ first."
    )
final_predictions = pd.read_parquet(final_prediction_path)

tfidf_predictions = None
if TFIDF_RUN_ID:
    tfidf_prediction_path = PREDICTIONS_DIR / f"{TFIDF_RUN_ID}_test.parquet"
    if not tfidf_prediction_path.exists():
        raise FileNotFoundError(f"Missing TF-IDF test predictions: {tfidf_prediction_path}")
    tfidf_predictions = pd.read_parquet(tfidf_prediction_path)

print(f"Train: {len(train_df):,} | Test: {len(test_df):,}")
print(f"Final model predictions: {len(final_predictions):,}")
print("Year breakdown is not available from the shared loader; the test split is 2019 only.")


Train: 60,000 | Test: 10,000
Final model predictions: 10,000
Year breakdown is not available from the shared loader; the test split is 2019 only.


## 2. Per-slice classification performance (C6)

Head/middle/tail groups and short/medium/long boundaries are fitted using training data only. Slice macro-F1 averages classes present in each slice. Party roles are keyword-detected proxies, not contract metadata; categories may overlap. No within-test year comparison is possible because the fixed test split contains only 2019 and the shared loader omits row-level years.


In [3]:
required_variables = ("final_predictions", "tfidf_predictions", "test_df", "train_df")
missing_variables = [name for name in required_variables if name not in globals()]
if missing_variables:
    raise RuntimeError(
        "Run the 'Load the fixed test data and prediction files' cell above first; "
        f"missing: {', '.join(missing_variables)}"
    )

slice_tables = []
prediction_sets = [("legal-bert-final", final_predictions)]
if tfidf_predictions is not None:
    prediction_sets.append(("tfidf-logreg", tfidf_predictions))
for model_name, predictions in prediction_sets:
    audit_frame = build_audit_frame(test_df, predictions, train_df)
    table = slice_performance(audit_frame)
    table.insert(0, "model", model_name)
    slice_tables.append(table)

slice_results = pd.concat(slice_tables, ignore_index=True)
display(slice_results)


,model,slice,value,support,accuracy,macro_f1,mean_predicted_confidence
0,legal-bert-final,frequency_group,head,6350,0.927874,0.932225,0.946976
1,legal-bert-final,frequency_group,middle,2367,0.797212,0.844423,0.854339
2,legal-bert-final,frequency_group,tail,1283,0.731099,0.716082,0.854226
3,legal-bert-final,length_group,long,2243,0.877842,0.786494,0.906426
4,legal-bert-final,length_group,medium,4797,0.870544,0.791305,0.916393
5,legal-bert-final,length_group,short,2960,0.868919,0.735375,0.912986
6,legal-bert-final,party_type,borrower_lender,394,0.883249,0.759413,0.925631
7,legal-bert-final,party_type,buyer_seller,60,0.900000,0.850794,0.918325
8,legal-bert-final,party_type,employer_employee,47,0.702128,0.583333,0.860627
9,legal-bert-final,party_type,landlord_tenant,118,0.694915,0.601367,0.862321


## 3. Bounded counterfactual classifier audit (C7–C8)

Generate one-feature counterfactuals from a fixed random sample of at most 1,000 test clauses. The sample bounds inference time; it is not a representative estimate of every possible wording change. Review transformed examples before interpreting aggregate flip rates. The model checkpoint path is recorded by the run config; on a local machine, copy the checkpoint from shared Drive or set `MODEL_CHECKPOINT_PATH` to its available location.


In [4]:
required_variables = ("RUNS_DIR", "FINAL_MODEL_RUN_ID", "test_df", "final_predictions")
missing_variables = [name for name in required_variables if name not in globals()]
if missing_variables:
    raise RuntimeError(
        "Run the data and prediction loading cell in section 1 before the counterfactual audit; "
        f"missing: {', '.join(missing_variables)}"
    )

run_config_path = RUNS_DIR / FINAL_MODEL_RUN_ID / "config.json"
run_config = json.loads(run_config_path.read_text())
MODEL_CHECKPOINT_PATH = str(
    PROJECT_ROOT
    / "models"
    / "ledgar_checkpoints"
    / "legal-bert_full_20261019-0916"
    / "checkpoint-11250"
)
if not Path(MODEL_CHECKPOINT_PATH).exists():
    raise FileNotFoundError(
        f"Final checkpoint is not available at {MODEL_CHECKPOINT_PATH}. "
        "Check that models/ledgar_checkpoints contains the final Legal-BERT checkpoint."
    )

predictor = HuggingFaceTextPredictor.from_pretrained(
    model_name=run_config["model_name"],
    checkpoint_path=MODEL_CHECKPOINT_PATH,
    revision=run_config.get("model_revision"),
    max_length=run_config["max_length"],
    batch_size=32,
)
COUNTERFACTUAL_SOURCE_LIMIT = 1_000
counterfactual_source = test_df.sample(
    n=min(COUNTERFACTUAL_SOURCE_LIMIT, len(test_df)),
    random_state=42,
)
counterfactual_pairs = generate_counterfactual_pairs(counterfactual_source)
print(
    f"Generated {len(counterfactual_pairs):,} counterfactual pair(s) "
    f"from {len(counterfactual_source):,} sampled test clauses"
)
counterfactual_results = audit_counterfactuals(counterfactual_pairs, predictor)
if not counterfactual_results.empty:
    expected_labels = final_predictions.set_index("idx")["pred"]
    saved_labels = counterfactual_results["idx"].map(expected_labels).astype(int)
    if not saved_labels.equals(counterfactual_results["original_label"]):
        raise ValueError("Checkpoint predictions do not match the saved final-model test predictions")
counterfactual_summary = summarize_counterfactuals(counterfactual_results)
display(counterfactual_summary)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Generated 236 counterfactual pair(s) from 1,000 sampled test clauses


,category,n_pairs,flip_rate,mean_confidence_delta
0,currency_usd_to_eur,21,0.00000,0.000054
1,jurisdiction_new_york_to_delaware,44,0.00000,-0.001088
2,jurisdiction_new_york_to_england,44,0.00000,-0.005081
3,jurisdiction_new_york_to_luxembourg,44,0.00000,-0.005998
4,pronoun_he_she,17,0.00000,0.000325
5,role_swap_borrower_lender,42,0.02381,0.001031
6,role_swap_buyer_seller,7,0.00000,-0.000317
7,role_swap_employer_employee,6,0.00000,-0.003945
8,role_swap_landlord_tenant,11,0.00000,-0.001162


## 4. Save the focused audit outputs

Review the generated counterfactual text before using aggregate results in the model card. The counterfactual results describe only the bounded sample and observed heuristic edits.


In [5]:
audit_results_dir = RESULTS_DIR / "audit"
audit_results_dir.mkdir(parents=True, exist_ok=True)
slice_results.to_csv(audit_results_dir / "slice_performance.csv", index=False)
pd.DataFrame(counterfactual_pairs).to_csv(audit_results_dir / "counterfactual_pairs.csv", index=False)
counterfactual_results.to_csv(audit_results_dir / "counterfactual_predictions.csv", index=False)
counterfactual_summary.to_csv(audit_results_dir / "counterfactual_summary.csv", index=False)
print(f"Saved focused audit tables under {audit_results_dir}")


Saved focused audit tables under /Users/brianooi/Desktop/Albert/Deep Learning/Projet_Deep_Learning/results/audit
